# LEGEND poster — configuration and setup

This notebook edits the **single saved master configuration**. Save here, then
run `poster_workbench.ipynb`. Defaults are a **provisional benchmark**, not a
calibrated LEGEND prediction. Natural abundances/masses are sourced; the enriched
92/8 preset is explicitly hypothetical. All physical/numerical edits go into
this JSON, not production modules. Historical pilot files remain under
`historical/` and `poster_workbench_assets/`.


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Open this notebook inside the WIMP-LEGEND checkout')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Image
from analyses.legend_poster import load_config, save_config, validate_config, resolve_profile
CONFIG_PATH = Path(os.environ.get('WIMP_POSTER_CONFIG', ROOT / 'analyses/legend_poster/config.json'))
print('Configuration:', CONFIG_PATH)


## Load and inspect sources

In [ ]:
configuration = load_config(CONFIG_PATH)
print(configuration['status'])
display(configuration['target']['mass_conversion'])
display(configuration['target']['presets'])
from analyses.legend_poster.models import target, isotope_weights
for name in configuration['target']['presets']:
    print(name)
    display([dict(A=i.A, nuclear_mass_GeV=i.mass_GeV, J=i.spin, fraction=f)
             for i, f in target(configuration, name)])
    display(isotope_weights(configuration, name))


## Edit a working dictionary

Change fields below or edit `analyses/legend_poster/config.json` directly.
Mass fractions mean **nuclear-ensemble** fractions as in `TargetComposition`;
bulk target counts include the sourced neutral atomic mass. Mass values and
atomic-to-nuclear conversion are checked for consistency. Unnormalized fractions
are rejected rather than silently changed.

Detector threshold is a scenario analysis cut. Component response overrides
select noise/Fano/efficiency/Lindhard parameters; observed residuals bypass them.
Background scales are explicit benchmark multipliers, separate from activities.


In [ ]:
from copy import deepcopy
working = deepcopy(configuration)

# Examples to UNCOMMENT and change deliberately:
# working['detector']['presets']['benchmark']['threshold_keV_obs'] = 0.2
# working['analysis']['exposures_kg_day'] = [365.25, 3652.5]
# working['signal']['masses_GeV'] = [3., 5., 10., 20.]
# working['background']['scenarios']['legend_poster']['components']['Ar39'] = {'enabled': False}
# working['background']['scenarios']['legend_poster']['overrides']['lar_veto_survival_probability'] = 0.02
# working['numerical']['bin_segments'] = [[0.03, 2., 0.005], [2., 15., 0.05]]

print('Selected scenarios:', working['analysis']['scenarios'])
for section in ('target', 'detector', 'background', 'signal', 'analysis', 'numerical', 'plotting'):
    print(section.upper())
    display(working[section])


## Validate and save

Saving is explicit and atomic. This is the only file the analysis reads for
working physical parameters. The cell below performs an unchanged round trip
unless you edited `working` above. Invalid changes raise an explanatory error.


In [ ]:
validated = validate_config(working)
save_config(validated, CONFIG_PATH)
assert load_config(CONFIG_PATH) == validated
from analyses.legend_poster.storage import digest
print('Saved configuration SHA-256:', digest(validated))


## Preview execution

`standard` uses the configured full mass grid and convergence requirements.
`smoke` is an inexpensive, explicitly coarse **numerical** profile from the
same master file; it still uses all physical components and isotope kernels.
Set `WIMP_POSTER_PROFILE=smoke` when executing notebooks for a quick test.
A standard first run can take minutes; subsequent runs reuse keyed outputs.


In [ ]:
profile = os.environ.get('WIMP_POSTER_PROFILE', validated['execution']['default_profile'])
resolved = resolve_profile(validated, profile)
from analyses.legend_poster.analysis import observed_edges
print('Profile:', profile, resolved['execution']['resolved_label'])
print('Masses [GeV]:', resolved['signal']['masses_GeV'])
print('Observed bins:', len(observed_edges(resolved))-1)
display(resolved['numerical'])
